# Telaah: SFT of Qwen2.5-7B-Instruct (QLoRA, Unsloth)

Teaches the pipeline's reading job: 5 retrieved pages in, a cited answer `[doc_id p.N]` or exactly `TIDAK DITEMUKAN` out
(see `sft/DESIGN.md`). Data: `train.jsonl` (1,213) and `val.jsonl` (111) from `sft/split_sft.py`, split by page.

**Settings (right panel):** Accelerator = **GPU T4 x2** (Unsloth uses one GPU; not P100), Internet = **On**,
input = your private `telaah-sft` dataset.

**Two modes, chosen automatically:**
- **Interactive session** (you press Run All in the editor) = **smoke test**: 20 training steps, timing, memory and an
  estimate of the full run. No export. Stop the session afterwards to save GPU quota.
- **Save Version > Save & Run All (Commit)** = **full run**: 1 epoch, validation loss every 50 steps, LoRA adapter zip
  and a GGUF Q4_K_M file for Ollama in the version's **Output** tab.

**Resume after a crash:** add the failed version's output as an extra input; the newest `checkpoints/checkpoint-*`
is copied and training resumes from it.

In [ ]:
# Cell 1: install. Kaggle's preinstalled torchao broke peft in the ColPali run (Oct 13), so remove it first;
# unsloth pulls a compatible version back in if it needs one.
!pip uninstall -y -q torchao
!pip install -q unsloth
import importlib.metadata as md
for p in ["torch", "transformers", "trl", "peft", "bitsandbytes", "unsloth", "unsloth_zoo", "datasets"]:
    try:
        print(f"{p:13} {md.version(p)}")
    except Exception:
        print(f"{p:13} not installed")

In [ ]:
# Cell 2: settings
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # Unsloth trains on one GPU; set before torch is imported
import glob, json, math, random, shutil, time

RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive")
FORCE_FULL = False                                 # True = full run even in an interactive session
SMOKE = (RUN_TYPE != "Batch") and not FORCE_FULL
SMOKE_STEPS = 20

MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_LEN = 8448                                     # smoke 1: median 5.3k, p99 ~8.3k, max 10.3k tokens
EPOCHS = 1
LR = 2e-4
BATCH, GRAD_ACC = 1, 8                             # effective batch 8
LORA_R = 16
EVAL_STEPS = SAVE_STEPS = 50
EXPORT_GGUF = True

WORK = "/kaggle/working"
CKPT = f"{WORK}/checkpoints"


def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found under /kaggle/input: add the telaah-sft dataset as input"
    return hits[0]


TRAIN, VAL = find("train.jsonl"), find("val.jsonl")
print(f"run type {RUN_TYPE} -> {'SMOKE TEST' if SMOKE else 'FULL RUN'}")
print("train:", TRAIN, "\nval:  ", VAL)

In [ ]:
# Cell 3: GPU and disk
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    pr = torch.cuda.get_device_properties(i)
    print(f"gpu {i}: {pr.name}, {pr.total_memory / 2**30:.1f} GB")
for d in ["/kaggle/working", "/tmp"]:
    u = shutil.disk_usage(d)
    print(f"disk {d}: {u.free / 2**30:.0f} GB free of {u.total / 2**30:.0f} GB")

In [ ]:
# Cell 4: model + LoRA
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_LEN, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
BF16 = is_bfloat16_supported()
print(f"bf16 {BF16}; GPU memory after load {torch.cuda.memory_allocated() / 2**30:.1f} GB")

In [ ]:
# Cell 5: data -> Qwen chat text (the same ChatML the pipeline sends through Ollama), token lengths
from datasets import Dataset


def load(path):
    out = []
    for line in open(path, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        roles = [m["role"] for m in r["messages"]]
        assert roles == ["system", "user", "assistant"], (r["cid"], roles)
        text = tokenizer.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=False)
        n = len(tokenizer(text, add_special_tokens=False)["input_ids"])
        out.append({"text": text, "n_tok": n, "cid": r["cid"], "mix": r["mix"], "kind": r["kind"],
                    "messages": r["messages"]})
    return out


train_rows, val_rows = load(TRAIN), load(VAL)
for name, rows in [("train", train_rows), ("val", val_rows)]:
    n = sorted(r["n_tok"] for r in rows)
    print(f"{name}: {len(rows)} examples, tokens min {n[0]} median {n[len(n)//2]} "
          f"p99 {n[int(.99 * len(n))]} max {n[-1]}; over MAX_LEN {sum(x > MAX_LEN for x in n)}")
train_rows = [r for r in train_rows if r["n_tok"] <= MAX_LEN]
val_rows = [r for r in val_rows if r["n_tok"] <= MAX_LEN]
print("kept", len(train_rows), "train,", len(val_rows), "val")
print("\n--- end of one example ---\n" + train_rows[0]["text"][-400:])

In [ ]:
# Cell 6: trainer, loss on the assistant answer only
import inspect
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

train_ds = Dataset.from_list([{"text": r["text"]} for r in (sorted(train_rows, key=lambda r: -r["n_tok"])[:160] if SMOKE else train_rows)])
val_ds = Dataset.from_list([{"text": r["text"]} for r in (val_rows[:8] if SMOKE else val_rows)])

cfg_keys = inspect.signature(SFTConfig.__init__).parameters
cfg = dict(
    output_dir=CKPT, per_device_train_batch_size=BATCH, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACC, num_train_epochs=EPOCHS, learning_rate=LR,
    warmup_steps=10, lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
    logging_steps=5, fp16=not BF16, bf16=BF16, seed=3407, report_to="none",
    dataset_text_field="text", packing=False,
    save_strategy="no" if SMOKE else "steps", save_steps=SAVE_STEPS, save_total_limit=2,
    eval_steps=SMOKE_STEPS if SMOKE else EVAL_STEPS)
cfg["max_length" if "max_length" in cfg_keys else "max_seq_length"] = MAX_LEN
cfg["eval_strategy" if "eval_strategy" in cfg_keys else "evaluation_strategy"] = "no" if SMOKE else "steps"
if SMOKE:
    cfg["max_steps"] = SMOKE_STEPS
    cfg["logging_steps"] = 1

tr_keys = inspect.signature(SFTTrainer.__init__).parameters
tok_kw = {"processing_class": tokenizer} if "processing_class" in tr_keys else {"tokenizer": tokenizer}
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=SFTConfig(**cfg), **tok_kw)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# Check the mask: the trained tokens must be the answer only (no "Konteks:", no system prompt).
ex = trainer.train_dataset[0]
kept = [t for t in ex["labels"] if t != -100]
target = tokenizer.decode(kept)
print(f"trained tokens {len(kept)} of {len(ex['input_ids'])}:", repr(target))
assert 0 < len(kept) < 0.2 * len(ex["input_ids"]) and "Konteks:" not in target, "loss mask looks wrong"

In [ ]:
# Cell 7: train (resumes from a previous version's checkpoint if one is attached as input)
resume = None
prev = sorted(glob.glob("/kaggle/input/**/checkpoints/checkpoint-*", recursive=True),
              key=lambda p: int(p.rsplit("-", 1)[1]))
if prev and not SMOKE:
    shutil.copytree(prev[-1], os.path.join(CKPT, os.path.basename(prev[-1])), dirs_exist_ok=True)
    resume = os.path.join(CKPT, os.path.basename(prev[-1]))
    print("resuming from", resume)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
result = trainer.train(resume_from_checkpoint=resume)
elapsed = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 2**30
print(f"\ntrain time {elapsed / 60:.1f} min, steps {trainer.state.global_step}, peak GPU memory {peak:.1f} GB")

if SMOKE:
    t1 = time.time()
    trainer.evaluate()
    eval_per_ex = (time.time() - t1) / len(val_ds)
    sec_step = elapsed / SMOKE_STEPS
    steps_full = math.ceil(len(train_rows) / (BATCH * GRAD_ACC)) * EPOCHS
    n_evals = steps_full // EVAL_STEPS + 1
    est = sec_step * steps_full + eval_per_ex * len(val_rows) * n_evals
    print(f"{sec_step:.1f} s/step; full run = {steps_full} steps + {n_evals} evals of {len(val_rows)} "
          f"-> about {est / 3600:.1f} h of training (+ ~30-45 min for export). Kaggle limit: 12 h per session.")

In [ ]:
# Cell 8: loss history
hist = trainer.state.log_history
json.dump(hist, open(f"{WORK}/train_log.json", "w"), indent=1)
for h in hist:
    if "eval_loss" in h:
        print(f"step {h['step']:4}  eval_loss {h['eval_loss']:.4f}")
tl = [h for h in hist if "loss" in h]
if tl:
    print(f"train loss first {tl[0]['loss']:.4f} -> last {tl[-1]['loss']:.4f}")

In [ ]:
# Cell 9: sanity answers on held-out pages (greedy, same prompt as the pipeline)
FastLanguageModel.for_inference(model)
rng = random.Random(0)
by_mix = {}
for r in val_rows:
    by_mix.setdefault(r["mix"], []).append(r)
pick = [rng.choice(v) for v in by_mix.values()]
if SMOKE:
    pick = pick[:2]
samples = []
for r in pick:
    prompt = tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=160, do_sample=False)
    pred = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gold = r["messages"][2]["content"]
    samples.append({"cid": r["cid"], "mix": r["mix"], "kind": r["kind"], "gold": gold, "pred": pred})
    print(f"[{r['cid']} {r['mix']}]\n  Q:    {r['messages'][1]['content'].rsplit('Pertanyaan:', 1)[-1].strip()}"
          f"\n  gold: {gold}\n  pred: {pred}\n")
with open(f"{WORK}/val_samples.jsonl", "w", encoding="utf-8") as f:
    for s in samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

In [ ]:
# Cell 10: save the LoRA adapter (small; enough to rebuild the model anywhere)
if SMOKE:
    print("smoke test: nothing saved. Stop the session now (power icon, top right) to save GPU quota.")
else:
    model.save_pretrained(f"{WORK}/lora_adapter")
    tokenizer.save_pretrained(f"{WORK}/lora_adapter")
    shutil.make_archive(f"{WORK}/lora_adapter", "zip", f"{WORK}/lora_adapter")
    shutil.rmtree(f"{WORK}/lora_adapter")
    print("saved lora_adapter.zip")

In [ ]:
# Cell 11: GGUF Q4_K_M for Ollama. Built in /tmp (the merged 16-bit model does not fit the 20 GB output folder),
# only the final file is copied to the output. If this fails, the adapter zip above is still there.
if not SMOKE and EXPORT_GGUF:
    try:
        model.save_pretrained_gguf("/tmp/gguf", tokenizer, quantization_method="q4_k_m")
        found = [p for p in glob.glob("/tmp/**/*.gguf", recursive=True) if "q4_k_m" in p.lower()]
        print("found:", found)
        assert found, "no Q4_K_M gguf produced"
        shutil.copy(found[0], f"{WORK}/qwen2.5-7b-telaah-sft-Q4_K_M.gguf")
        print("saved qwen2.5-7b-telaah-sft-Q4_K_M.gguf")
    except Exception as e:
        print("GGUF export FAILED:", repr(e))
        print("The adapter zip is saved; the merge + conversion can be redone in a short second run.")

In [ ]:
# Cell 12: clean up checkpoints after a finished run, list the output
if not SMOKE and os.path.exists(f"{WORK}/lora_adapter.zip"):
    shutil.rmtree(CKPT, ignore_errors=True)
for p in sorted(glob.glob(f"{WORK}/*")):
    size = os.path.getsize(p) if os.path.isfile(p) else sum(
        os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
    print(f"{size / 2**20:10.1f} MB  {os.path.basename(p)}")